### `benchmark.py`启动命令说明

In [ ]:
cd /data/wanghanzhen/Projects/MTP/NIPS26/FlashMTP_v1.1
source .venv/bin/activate
export DATASET='alpaca' SAMPLES=50 DT='h100' BS=1 BLOCK=16
CUDA_VISIBLE_DEVICES=5 python3 evaluation/benchmark.py \
  --model-name-or-path $WHZ_DIR/models/Qwen/Qwen3-8B \
  --draft-name-or-path /data/wanghanzhen/Projects/MTP/NIPS26/FlashMTP_v1.1/cache/models/flashmtp_h100_prefix_condition_fuse18_sample_40000_think_off_nlayers5_block_16_maxlen4096_epochs8_tlmh0_lp1_tmc0_an1_w1mse0/epoch_8_step_39816 \
  --dataset ${DATASET} \
  --max-new-tokens 512 \
  --batch-size ${BS} \
  --max-samples ${SAMPLES} > ./log/${DATASET}_flashmtp_v1.1_noise_block_${BLOCK}_bs${BS}_${DT}_${SAMPLES}.log

### 开启草稿树
trunc-thres：草稿链小于thres直接截断
expand-thres and entropy-ratio: prob小于p_expand且top-wd entropy大于ratio：扩展

In [ ]:
# trunc and expand

cd /data/wanghanzhen/Projects/MTP/NIPS26/FlashMTP_v1.1
source .venv/bin/activate
CUDA_VISIBLE_DEVICES=1 export DATASET='mt-bench' SAMPLES=50 DT='h100' BS=1 BLOCK=16
python evaluation/benchmark.py \
  --draft-name-or-path /data/wanghanzhen/Projects/MTP/NIPS26/FlashMTP_v1.1/cache/models/flashmtp_qz_prefix_condition_fuse_middle_16_feature_sample_900000_think_off_nlayers5_block_16_gamma_7_maxlen4096_epochs8_tlmh0_lp1 \
  --dataset ${DATASET} \
  --max-new-tokens 512 \
  --batch-size ${BS} \
  --max-samples ${SAMPLES} \
  --use-draft-tree \
  --draft-tree-trunc-thres 0.2 \
  --draft-tree-expand-thres 0.5 \
  --draft-tree-width 4 \
  --draft-tree-entropy-ratio 0.6 > ./log/${DATASET}_flashmtp_v1.1_tree_block_entropy_0.6_${BLOCK}_bs${BS}_${DT}_${SAMPLES}.log

### `spec_profile.py` 查看大小模型预测细节信息

In [ ]:
torchrun --nproc_per_node=1 evaluation/spec_profile.py \
  --draft-name-or-path /data/wanghanzhen/Projects/MTP/NIPS26/FlashMTP_v1.1/cache/models/flashmtp_h100_prefix_condition_fuse18_sample_40000_think_off_nlayers5_block_16_maxlen4096_epochs6_tlmh0_lp0_tmc1/epoch_6_step_29862 \
  --dataset multi_alpacanews_e \
  --max-samples 2 \
  --max-new-tokens 128 \
  --temperature 0.0 \
  --output-jsonl log/spec_profile_flashmtp_epoch6.jsonl